# §3 Main Table — Headline FID-50K Experiments (n=5)

**Goal.** Produce the headline results table for the final paper:
- Vanilla ARPG-L at step counts {8, 12, 16, 24, 32}, seeds {0, 1, 2, 3, 4}
- RTR (Random Token Rejection, ρ=0.5) at the same step counts and seeds
- 50 total configurations → multi-seed mean ± std for every cell

**Persistence / resumability.**
- All NPZs are saved to Google Drive at `MyDrive/ARPG-assets/results/final-paper/main-table/`.
- The master CSV `main-table/results.csv` is the single source of truth: every successful config writes a row immediately.
- If Colab disconnects, just rerun the notebook end-to-end — the main loop skips any config already in the master CSV.
- Known seed-0 results from Phase 3 / Phase 5 are pre-loaded into the master CSV on first run, so we don’t re-sample them.

**Order.** Configs run in priority order: 16 steps first (headline regime), then 8, then 32, then 12 and 24 (fill-in points for the scaling-law curve). This way the abstract numbers land in the first overnight session.

**Compute.** ~29 hours of A100 time total, ~13 hours per overnight Colab Pro session, so ~3 sessions to finish.

**Disk.** Each NPZ is ~10 GB. 39 new NPZs ≈ 390 GB of new Drive data. Make sure Drive has space.

## 1. Mount Drive and set up paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Drive root — contains weights/, eval/, external/, results/
DRIVE_ROOT = Path('/content/drive/MyDrive/ARPG-assets')

# Output location for the §3 main table
RESULTS_ROOT = DRIVE_ROOT / 'results' / 'final-paper' / 'main-table'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CSV_PATH = RESULTS_ROOT / 'results.csv'
VANILLA_NPZ_DIR = RESULTS_ROOT / 'vanilla'
RTR_NPZ_DIR = RESULTS_ROOT / 'rtr'
VANILLA_NPZ_DIR.mkdir(parents=True, exist_ok=True)
RTR_NPZ_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = RESULTS_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Repo + local sampling scratch space (local disk is much faster than Drive for I/O)
REPO_LOCAL = Path('/content/ARPG-main')
LOCAL_SAMPLE_DIR = Path('/content/samples')
LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# Fixed assets
REF_NPZ = DRIVE_ROOT / 'eval' / 'VIRTUAL_imagenet256_labeled.npz'
ARPG_CKPT = DRIVE_ROOT / 'weights' / 'arpg_300m.pt'
VQ_CKPT = DRIVE_ROOT / 'weights' / 'vq_ds16_c2i.pt'
GD_REPO = DRIVE_ROOT / 'external' / 'guided-diffusion'


# Qualitative-samples output (for paper figures)
SAMPLES_DIR = RESULTS_ROOT / 'samples'
SAMPLES_GRIDS_DIR = SAMPLES_DIR / 'grids'
SAMPLES_INDIVIDUAL_DIR = SAMPLES_DIR / 'individual'
SAMPLES_GRIDS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_INDIVIDUAL_DIR.mkdir(parents=True, exist_ok=True)

# Rejection tracker output (small JSON + heatmap PNG per RTR config; kept on Drive)
REJECTION_LOGS_DIR = RESULTS_ROOT / 'rejection-logs'
REJECTION_LOGS_DIR.mkdir(parents=True, exist_ok=True)

# NPZ persistence policy.
# False = keep only FID metrics + 8 qualitative PNGs + rejection JSON; delete the 10 GB NPZ.
# True  = also archive every NPZ to Drive (390 GB total for the main table — needs ~2 TB plan).
KEEP_NPZ_ON_DRIVE = False

print(f'Drive root  : {DRIVE_ROOT}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Master CSV  : {CSV_PATH}')
print(f'Repo (local): {REPO_LOCAL}')
print(f'Sample (loc): {LOCAL_SAMPLE_DIR}')
print(f'Samples (paper figures): {SAMPLES_DIR}')

## 2. Clone repo and verify all assets exist

In [ ]:
# Private fork containing the random-deferral support (commit 605038a or later)
REPO_URL = 'https://github.com/rshahbazov23/comp447-arpg-private.git'

# If the repo is private, set a GitHub Personal Access Token here so the clone
# works without an interactive prompt. Create one at:
#   https://github.com/settings/tokens  (classic, scope: repo)
GITHUB_TOKEN = None  # e.g. 'ghp_...'

import subprocess

def _clone_url(url, token):
    if token and url.startswith('https://github.com/'):
        return url.replace('https://', f'https://{token}@')
    return url

# --- 1. Clone / update the ARPG fork ---------------------------------------
if not REPO_LOCAL.exists():
    print(f'Cloning {REPO_URL} → {REPO_LOCAL}')
    subprocess.run(['git', 'clone', _clone_url(REPO_URL, GITHUB_TOKEN), str(REPO_LOCAL)], check=True)
else:
    print(f'Repo already present, pulling latest')
    subprocess.run(['git', '-C', str(REPO_LOCAL), 'pull'], check=True)

# --- 2. Auto-setup assets on Drive (download if missing) -------------------
# Total: ~4 GB. Downloads to Drive so subsequent sessions don’t re-download.

ASSET_URLS = {
    REF_NPZ:   'https://openaipublic.blob.core.windows.net/diffusion/jul-2021/ref_batches/imagenet/256/VIRTUAL_imagenet256_labeled.npz',
    ARPG_CKPT: 'https://huggingface.co/hp-l33/ARPG/resolve/main/arpg_300m.pt',
    VQ_CKPT:   'https://huggingface.co/FoundationVision/LlamaGen/resolve/main/vq_ds16_c2i.pt',
}

# Quick check of a few plausible alternate locations on Drive (avoids slow rglob)
ALT_LOCATIONS = [
    Path('/content/drive/MyDrive/eval'),
    Path('/content/drive/MyDrive/weights'),
    Path('/content/drive/MyDrive/ARPG/eval'),
    Path('/content/drive/MyDrive/ARPG/weights'),
    Path('/content/drive/MyDrive/ARPG-main/eval'),
    Path('/content/drive/MyDrive/ARPG-main/weights'),
]

import shutil

def find_alt(filename):
    for root in ALT_LOCATIONS:
        cand = root / filename
        if cand.exists():
            return cand
    return None

for target, url in ASSET_URLS.items():
    if target.exists():
        size_gb = target.stat().st_size / 1e9
        print(f'OK   {target.name:<40} ({size_gb:.2f} GB)')
        continue

    alt = find_alt(target.name)
    if alt:
        print(f'Found {target.name} at {alt} — copying to {target}')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(alt, target)
        continue

    print(f'Downloading {target.name} → {target}')
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['wget', '--show-progress', '-O', str(target), url], check=True)
    size_gb = target.stat().st_size / 1e9
    print(f'   downloaded ({size_gb:.2f} GB)')

# --- 3. guided-diffusion repo ----------------------------------------------
if not GD_REPO.exists():
    alt_gd = None
    for root in [Path('/content/drive/MyDrive/external/guided-diffusion'),
                 Path('/content/drive/MyDrive/guided-diffusion'),
                 Path('/content/drive/MyDrive/ARPG/external/guided-diffusion')]:
        if root.exists() and (root / 'evaluations' / 'evaluator.py').exists():
            alt_gd = root
            break
    if alt_gd:
        print(f'Found guided-diffusion at {alt_gd} — copying to {GD_REPO}')
        GD_REPO.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(alt_gd, GD_REPO)
    else:
        print(f'Cloning guided-diffusion → {GD_REPO}')
        GD_REPO.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', 'https://github.com/openai/guided-diffusion.git', str(GD_REPO)], check=True)

# --- 4. Final sanity check -------------------------------------------------
required = [
    (REF_NPZ,   'ImageNet reference batch'),
    (ARPG_CKPT, 'ARPG-L pretrained checkpoint'),
    (VQ_CKPT,   'LlamaGen VQ tokenizer'),
    (GD_REPO,   'guided-diffusion repo'),
    (GD_REPO / 'evaluations' / 'evaluator.py', 'guided-diffusion evaluator'),
    (REPO_LOCAL / 'sample_c2i_ddp.py', 'ARPG sampling script'),
    (REPO_LOCAL / 'models' / 'arpg.py', 'ARPG model'),
    (REPO_LOCAL / 'models' / 'confidence.py', 'Confidence metrics module'),
]
for p, name in required:
    if not p.exists():
        raise FileNotFoundError(f'MISSING: {name} — expected at {p}')
print('\nAll assets present.')

# --- 5. Confirm random support in the cloned fork --------------------------
conf_src = (REPO_LOCAL / 'models' / 'confidence.py').read_text()
if 'random_score' not in conf_src:
    raise RuntimeError('confidence.py does not include random selection. '
                       'Make sure the repo is on a branch with random-deferral support '
                       '(commit 605038a or later).')
print('Random-deferral support confirmed in confidence.py.')


## 3. Install Python dependencies

In [ ]:
# Most of these are already in the Colab base image; -q makes pip quiet.
subprocess.run(['pip', 'install', '-q',
    'einops',
    'transformers',
    'scipy',
    'tensorflow',           # guided-diffusion evaluator needs TF
    'pandas',
], check=True)

# Quick import sanity check
import torch, einops, transformers, scipy, pandas as pd
print(f'torch        : {torch.__version__}, CUDA {torch.version.cuda}, GPU {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
print(f'einops       : {einops.__version__}')
print(f'transformers : {transformers.__version__}')
print(f'scipy        : {scipy.__version__}')
print(f'pandas       : {pd.__version__}')
assert torch.cuda.is_available(), 'No CUDA device! Switch the Colab runtime to GPU.'

## 4. Build the config matrix and initialise the master CSV

Pre-loads known seed-0 results from Phase 3 / Phase 5 Item 1 so we don’t re-sample them.
Attempts to read `final-paper/multi-seed/multiseed-results.csv` to backfill the seed-1/2 vanilla results from Phase 5 Item 2.

In [ ]:
import pandas as pd
from datetime import datetime

# Priority order: headline regime first, fill-in points last
STEP_COUNTS_ORDER = [16, 8, 32, 12, 24]
SEEDS = [0, 1, 2, 3, 4]

# RTR hyperparameters (Random Token Rejection)
# - cap rho = 0.5 is the headline from Phase 5 / cap-saturation finding
# - threshold tau = 2.0 is set above the max possible random score so the cap branch
#   always fires; tau is empirically dead in our setting (Phase 1+2+5 all confirm)
RTR_CAP = 0.5
RTR_THRESHOLD = 2.0
RTR_METRIC = 'random'

# Canonical ImageNet classes used for qualitative figures (matches the
# ARPG paper's HuggingFace quick-start: golden retriever, otter, panda,
# snail, Arabian camel, volcano, monarch butterfly, valley).
QUALITATIVE_CLASSES = [207, 360, 388, 113, 355, 980, 323, 979]

def make_configs():
    out = []
    for step in STEP_COUNTS_ORDER:
        for mode in ['vanilla', 'rtr']:
            for seed in SEEDS:
                out.append({'mode': mode, 'step': step, 'seed': seed})
    return out

CONFIGS = make_configs()
print(f'Config matrix: {len(CONFIGS)} configs (5 steps × 2 modes × 5 seeds)')

# Known prior results from the experiments log
PRIOR_RESULTS = [
    # Phase 3 FID-50K vanilla seed 0
    {'mode': 'vanilla', 'step': 8,  'seed': 0, 'fid': 10.626, 'source': 'phase3'},
    {'mode': 'vanilla', 'step': 16, 'seed': 0, 'fid': 3.609,  'source': 'phase3'},
    {'mode': 'vanilla', 'step': 32, 'seed': 0, 'fid': 2.384,  'source': 'phase3'},
    # Phase 5 Item 1 — random-deferral seed 0
    {'mode': 'rtr',     'step': 8,  'seed': 0, 'fid': 4.851,  'source': 'phase5-item1'},
    {'mode': 'rtr',     'step': 16, 'seed': 0, 'fid': 2.633,  'source': 'phase5-item1'},
]

# Backfill from Phase 5 Item 2 (multi-seed vanilla seeds 1, 2) if the CSV is present
MULTISEED_CSV = DRIVE_ROOT / 'results' / 'final-paper' / 'multi-seed' / 'multiseed-results.csv'
if MULTISEED_CSV.exists():
    try:
        df_ms = pd.read_csv(MULTISEED_CSV)
        print(f'Loaded multiseed CSV — columns: {list(df_ms.columns)}')
        # Schema is flexible — print head so the user can verify
        print(df_ms.head())
        # Only consume vanilla rows (margin rows are not relevant to §3 RTR table)
        for _, row in df_ms.iterrows():
            mode = str(row.get('mode', '')).lower()
            if mode == 'vanilla':
                PRIOR_RESULTS.append({
                    'mode': 'vanilla',
                    'step': int(row['step']),
                    'seed': int(row['seed']),
                    'fid': float(row['fid']),
                    'source': 'phase5-item2',
                })
    except Exception as e:
        print(f'Could not auto-load multiseed CSV ({e}). Will re-run those configs.')
else:
    print(f'No multiseed CSV at {MULTISEED_CSV} — will re-run vanilla seeds 1, 2 if needed.')

# Deduplicate PRIOR_RESULTS
seen = set()
deduped = []
for r in PRIOR_RESULTS:
    key = (r['mode'], r['step'], r['seed'])
    if key not in seen:
        seen.add(key)
        deduped.append(r)
PRIOR_RESULTS = deduped
print(f'Total unique prior results to seed CSV: {len(PRIOR_RESULTS)}')

# Initialise (or extend) master CSV
if not CSV_PATH.exists():
    df_master = pd.DataFrame(PRIOR_RESULTS)
    df_master['timestamp'] = datetime.now().isoformat()
    df_master.to_csv(CSV_PATH, index=False)
    print(f'Initialised master CSV with {len(df_master)} prior rows')
else:
    df_master = pd.read_csv(CSV_PATH)
    existing_keys = set((r['mode'], int(r['step']), int(r['seed'])) for _, r in df_master.iterrows())
    new_rows = [r for r in PRIOR_RESULTS
                if (r['mode'], r['step'], r['seed']) not in existing_keys]
    if new_rows:
        for r in new_rows:
            r['timestamp'] = datetime.now().isoformat()
        df_master = pd.concat([df_master, pd.DataFrame(new_rows)], ignore_index=True)
        df_master.to_csv(CSV_PATH, index=False)
    print(f'Master CSV has {len(df_master)} rows; {len(new_rows)} backfilled this session.')

# Show what’s already done vs what’s left
done_keys = set((r['mode'], int(r['step']), int(r['seed'])) for _, r in df_master.iterrows())
remaining = [c for c in CONFIGS if (c['mode'], c['step'], c['seed']) not in done_keys]
print(f'\nProgress: {len(CONFIGS) - len(remaining)}/{len(CONFIGS)} done, {len(remaining)} remaining\n')
if remaining:
    print('Next 10 configs to run:')
    for c in remaining[:10]:
        print(f'  {c}')

## 5. Helper functions (sampling, FID eval, Drive sync, CSV update)

In [ ]:
import re, shutil, time, traceback

def config_to_folder_name(cfg):
    """Reproduces the exact folder-name convention from sample_c2i_ddp.py:114-122."""
    base = (
        f'ARPG-L-arpg_300m-size-256-size-256-VQ-16-'
        f'topk-0-topp-1.0-temperature-1.0-cfg-5.0-cfg-schedule-linear-'
        f'sample-schedule-arccos-step-{cfg["step"]}-seed-{cfg["seed"]}'
    )
    if cfg['mode'] == 'rtr':
        base += f'-mode-rejection-metric-{RTR_METRIC}-tau-{RTR_THRESHOLD}-cap-{RTR_CAP}'
    return base


def is_done(cfg, df_master):
    mask = (
        (df_master['mode'] == cfg['mode'])
        & (df_master['step'].astype(int) == cfg['step'])
        & (df_master['seed'].astype(int) == cfg['seed'])
    )
    return bool(mask.any())


def cleanup_local_samples():
    """Wipe /content/samples after we’ve synced the NPZ to Drive."""
    if LOCAL_SAMPLE_DIR.exists():
        shutil.rmtree(LOCAL_SAMPLE_DIR)
    LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)


def run_sampling(cfg, log_handle=None):
    """Sample 50K images to local disk. Returns the path to the NPZ."""
    folder = config_to_folder_name(cfg)
    cmd = [
        'torchrun', '--nnodes=1', '--nproc_per_node=1',
        'sample_c2i_ddp.py',
        '--gpt-model', 'ARPG-L',
        '--gpt-ckpt', str(ARPG_CKPT),
        '--vq-ckpt', str(VQ_CKPT),
        '--sample-schedule', 'arccos',
        '--cfg-schedule', 'linear',
        '--cfg-scale', '5.0',
        '--step', str(cfg['step']),
        '--per-proc-batch-size', '64',
        '--num-fid-samples', '50000',
        '--global-seed', str(cfg['seed']),
        '--sample-dir', str(LOCAL_SAMPLE_DIR),
        '--no-compile',
        '--precision', 'bf16',
    ]
    if cfg['mode'] == 'rtr':
        folder_for_log = config_to_folder_name(cfg)
        cmd += [
            '--rejection-mode', 'rejection',
            '--confidence-metric', RTR_METRIC,
            '--rejection-threshold', str(RTR_THRESHOLD),
            '--max-reject-rate', str(RTR_CAP),
            '--log-json', str(REJECTION_LOGS_DIR / f'{folder_for_log}.json'),
        ]

    print(f'  Sampling cmd: {" ".join(cmd)}')
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=str(REPO_LOCAL),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    last_print = t0
    for line in proc.stdout:
        if log_handle is not None:
            log_handle.write(line)
            log_handle.flush()
        # Print a heartbeat every 60 s so Colab knows we’re alive
        now = time.time()
        if now - last_print > 60:
            print(f'    [{(now-t0)/60:.1f} min] {line.rstrip()[:120]}')
            last_print = now
    proc.wait()
    elapsed = time.time() - t0
    print(f'  Sampling done in {elapsed/60:.1f} min (exit {proc.returncode})')

    if proc.returncode != 0:
        raise RuntimeError(f'Sampling failed for {cfg}: exit {proc.returncode}')

    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'
    if not local_npz.exists():
        raise FileNotFoundError(f'Expected NPZ not produced: {local_npz}')
    return local_npz


def sync_npz_to_drive(local_npz, cfg):
    if not KEEP_NPZ_ON_DRIVE:
        return None  # NPZ stays only on local disk and gets wiped after eval
    target_dir = VANILLA_NPZ_DIR if cfg['mode'] == 'vanilla' else RTR_NPZ_DIR
    drive_npz = target_dir / local_npz.name
    size_gb = local_npz.stat().st_size / 1e9
    print(f'  Syncing {local_npz.name} ({size_gb:.1f} GB) → Drive…')
    t0 = time.time()
    shutil.copy2(local_npz, drive_npz)
    print(f'  Drive sync done in {time.time() - t0:.1f} s')
    return drive_npz


_METRIC_LINE = re.compile(r'^\s*(FID|sFID|Inception Score|Precision|Recall)\s*:\s*([0-9.eE+\-]+)')

def evaluate_fid(local_npz, log_handle=None):
    """Run guided-diffusion’s evaluator. Returns dict with fid / inception_score / sfid / precision / recall."""
    cmd = ['python', 'evaluations/evaluator.py', str(REF_NPZ), str(local_npz)]
    print(f'  FID-eval cmd: cd {GD_REPO} && {" ".join(cmd)}')
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=str(GD_REPO), capture_output=True, text=True)
    elapsed = time.time() - t0
    print(f'  FID eval done in {elapsed/60:.1f} min (exit {proc.returncode})')

    if log_handle is not None:
        log_handle.write('--- evaluator stdout ---\n')
        log_handle.write(proc.stdout)
        log_handle.write('\n--- evaluator stderr ---\n')
        log_handle.write(proc.stderr)
        log_handle.flush()

    if proc.returncode != 0:
        print('STDOUT (tail):', proc.stdout[-2000:])
        print('STDERR (tail):', proc.stderr[-2000:])
        raise RuntimeError(f'FID eval failed: exit {proc.returncode}')

    metrics = {}
    for line in proc.stdout.splitlines():
        m = _METRIC_LINE.match(line)
        if m:
            key = m.group(1).lower().replace(' ', '_')
            metrics[key] = float(m.group(2))
    if 'fid' not in metrics:
        print('STDOUT:', proc.stdout)
        raise ValueError('Could not parse FID from evaluator output')
    return metrics


def append_result(cfg, metrics, npz_path):
    """Atomic-ish append to master CSV: read → concat → write.
    Drive is the single source of truth, so we always re-read before writing."""
    df = pd.read_csv(CSV_PATH)
    row = {
        'mode': cfg['mode'],
        'step': cfg['step'],
        'seed': cfg['seed'],
        'fid': metrics.get('fid'),
        'inception_score': metrics.get('inception_score'),
        'sfid': metrics.get('sfid'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'npz_path': str(npz_path) if npz_path is not None else '(not kept)',
        'source': 'main-table-run',
        'timestamp': datetime.now().isoformat(),
    }
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(CSV_PATH, index=False)
    return df


def save_qualitative_samples(local_npz, cfg, num_classes=1000):
    """Extract one sample per QUALITATIVE_CLASSES from the NPZ, save individual
    PNGs + a 2x4 grid composite. Output: SAMPLES_INDIVIDUAL_DIR/{folder}/class*.png
    plus SAMPLES_GRIDS_DIR/{folder}_grid.png.

    Assumes single-GPU sampling with build_class_schedule(N, num_classes) tiling:
    sample at NPZ index i is conditioned on class (i % num_classes), so the
    first sample for class C is at index C.
    """
    import numpy as np
    from PIL import Image

    npz = np.load(str(local_npz))
    arr = npz['arr_0']  # (N, H, W, 3) uint8
    folder = config_to_folder_name(cfg)
    indiv_dir = SAMPLES_INDIVIDUAL_DIR / folder
    indiv_dir.mkdir(parents=True, exist_ok=True)

    pils = []
    for cls in QUALITATIVE_CLASSES:
        idx = cls  # first occurrence of class C is at NPZ index C
        if idx >= arr.shape[0]:
            print(f'    WARN class {cls} idx {idx} out of range')
            continue
        pil = Image.fromarray(arr[idx])
        pil.save(indiv_dir / f'class{cls:03d}.png')
        pils.append((cls, pil))

    if len(pils) == len(QUALITATIVE_CLASSES):
        w, h = pils[0][1].size
        ncols = 4
        nrows = (len(pils) + ncols - 1) // ncols
        grid = Image.new('RGB', (w * ncols, h * nrows), color=(255, 255, 255))
        for i, (_, pil) in enumerate(pils):
            r, c = divmod(i, ncols)
            grid.paste(pil, (c * w, r * h))
        grid_path = SAMPLES_GRIDS_DIR / f'{folder}_grid.png'
        grid.save(grid_path)
        print(f'  Saved {len(pils)} class samples + grid → {grid_path.name}')
    else:
        print(f'  Saved {len(pils)} class samples (grid skipped — incomplete)')


print('Helpers loaded.')

## 6. Main loop — resumable, skips anything already in the master CSV

Each iteration: re-read CSV → skip if done → sample to local disk → sync NPZ to Drive → evaluate FID → append row to CSV → cleanup local disk.

On failure: log the exception, keep going to the next config.

**This cell is safe to interrupt at any time** — progress is saved after each config.

In [ ]:
failures = []
started = datetime.now()

for i, cfg in enumerate(CONFIGS, 1):
    print(f'\n{"="*70}\n[{i}/{len(CONFIGS)}] config={cfg}\n{"="*70}')

    # Always re-read CSV at the top of the loop (Drive is the source of truth)
    df_master = pd.read_csv(CSV_PATH)
    if is_done(cfg, df_master):
        print('  SKIP (already in master CSV)')
        continue

    # If the NPZ is already on Drive from a previous (incomplete) run, reuse it instead
    # of re-sampling. Saves 30+ minutes per recovered config.
    folder = config_to_folder_name(cfg)
    target_dir = VANILLA_NPZ_DIR if cfg['mode'] == 'vanilla' else RTR_NPZ_DIR
    drive_npz_existing = target_dir / f'{folder}.npz'
    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'

    log_path = LOG_DIR / f'{folder}.log'
    try:
        with open(log_path, 'w') as log_h:
            if drive_npz_existing.exists():
                print(f'  NPZ already on Drive ({drive_npz_existing.stat().st_size/1e9:.1f} GB) — re-using for eval')
                shutil.copy2(drive_npz_existing, local_npz)
                drive_npz = drive_npz_existing
            else:
                local_npz = run_sampling(cfg, log_handle=log_h)
                drive_npz = sync_npz_to_drive(local_npz, cfg)  # None if KEEP_NPZ_ON_DRIVE=False
                if drive_npz is None:
                    print(f'  NPZ kept on local disk only ({local_npz.stat().st_size/1e9:.1f} GB) — will be wiped after eval')

            metrics = evaluate_fid(local_npz, log_handle=log_h)
            append_result(cfg, metrics, drive_npz)

            # Extract qualitative samples for paper figures (cheap, ~3s)
            try:
                save_qualitative_samples(local_npz, cfg)
            except Exception as e:
                print(f'  Qualitative samples skipped: {e}')

        print(f'  DONE   FID={metrics["fid"]:.4f}   IS={metrics.get("inception_score", ""):.2f}   Prec={metrics.get("precision", ""):.3f}   Rec={metrics.get("recall", ""):.3f}')
    except Exception as e:
        print(f'  FAILED: {e}')
        traceback.print_exc()
        failures.append({'config': cfg, 'error': str(e), 'log': str(log_path)})
    finally:
        cleanup_local_samples()

    # Elapsed-time report (so you can decide if Colab is about to disconnect)
    elapsed = (datetime.now() - started).total_seconds() / 3600
    print(f'  Session elapsed: {elapsed:.2f} h')

print(f'\n\n{"="*70}\nSESSION COMPLETE\n{"="*70}')
print(f'Configs attempted: {len(CONFIGS)}')
print(f'Failures: {len(failures)}')
for f in failures:
    print(f'  {f["config"]}  →  {f["error"]}  (log: {f["log"]})')

## 7. Summary — mean ± std for the headline table

In [ ]:
import numpy as np

df = pd.read_csv(CSV_PATH)
df['step'] = df['step'].astype(int)
df['seed'] = df['seed'].astype(int)
print(f'Master CSV: {len(df)} rows total\n')

summary = (
    df.groupby(['mode', 'step'])['fid']
      .agg(['mean', 'std', 'count'])
      .round(4)
      .sort_index()
)
print('FID-50K — mean ± std (n)')
print(summary.to_string())

# Compute headline percentages: RTR vs vanilla at each step count
print('\n\nHeadline deltas (RTR vs vanilla, multi-seed mean):')
for step in sorted(df['step'].unique()):
    v = df[(df['mode']=='vanilla') & (df['step']==step)]['fid']
    r = df[(df['mode']=='rtr')     & (df['step']==step)]['fid']
    if len(v) > 0 and len(r) > 0:
        v_mean, r_mean = v.mean(), r.mean()
        delta = r_mean - v_mean
        pct = 100 * delta / v_mean
        print(f'  {step:>3} steps: vanilla={v_mean:.4f} (n={len(v)})  RTR={r_mean:.4f} (n={len(r)})  '
              f'Δ={delta:+.4f}  ({pct:+.2f}%)')

# Gap-closure (16 → 32 step regime)
v16 = df[(df['mode']=='vanilla') & (df['step']==16)]['fid'].mean()
v32 = df[(df['mode']=='vanilla') & (df['step']==32)]['fid'].mean()
r16 = df[(df['mode']=='rtr')     & (df['step']==16)]['fid'].mean()
if all(not np.isnan(x) for x in [v16, v32, r16]):
    gap_closure = (v16 - r16) / (v16 - v32)
    print(f'\nGap closure: RTR@16 closes {100*gap_closure:.1f}% of the 16→32-step vanilla quality gap')
    print(f'  vanilla@16 = {v16:.4f}')
    print(f'  RTR@16     = {r16:.4f}')
    print(f'  vanilla@32 = {v32:.4f}')

# Write summary CSV alongside the master
summary_path = RESULTS_ROOT / 'summary.csv'
summary.to_csv(summary_path)
print(f'\nWrote summary table: {summary_path}')

## 8. Compose vanilla-vs-RTR comparison figures for the paper

For each (step, seed) pair where both vanilla and RTR grids exist, stack them side-by-side with FID labels. Output saved to `samples/comparisons/`.

Safe to re-run any time. Only generates comparisons for configs that have grids on Drive.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw, ImageFont

COMPARISONS_DIR = SAMPLES_DIR / 'comparisons'
COMPARISONS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
df['step'] = df['step'].astype(int)
df['seed'] = df['seed'].astype(int)

try:
    FONT = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 28)
    FONT_SMALL = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 20)
except Exception:
    FONT = ImageFont.load_default()
    FONT_SMALL = ImageFont.load_default()

def fid_for(mode, step, seed):
    row = df[(df['mode']==mode) & (df['step']==step) & (df['seed']==seed)]
    if len(row) == 0:
        return None
    return float(row.iloc[0]['fid'])

def find_grid(mode, step, seed):
    cfg = {'mode': mode, 'step': step, 'seed': seed}
    name = config_to_folder_name(cfg) + '_grid.png'
    candidate = SAMPLES_GRIDS_DIR / name
    return candidate if candidate.exists() else None

made = 0
skipped = 0
for step in sorted(df['step'].unique()):
    for seed in sorted(df[df['step']==step]['seed'].unique()):
        van_grid_path = find_grid('vanilla', int(step), int(seed))
        rtr_grid_path = find_grid('rtr',     int(step), int(seed))
        if van_grid_path is None or rtr_grid_path is None:
            skipped += 1
            continue

        van_grid = Image.open(van_grid_path)
        rtr_grid = Image.open(rtr_grid_path)
        w, h = van_grid.size
        gap = 30
        header = 70
        out_w = w * 2 + gap
        out_h = h + header + 20
        canvas = Image.new('RGB', (out_w, out_h), color=(255, 255, 255))
        canvas.paste(van_grid, (0, header))
        canvas.paste(rtr_grid, (w + gap, header))

        draw = ImageDraw.Draw(canvas)
        van_fid = fid_for('vanilla', int(step), int(seed))
        rtr_fid = fid_for('rtr',     int(step), int(seed))
        van_label = f'Vanilla — {step} steps, seed {seed}'
        rtr_label = f'RTR — {step} steps, seed {seed}'
        if van_fid is not None:
            van_label += f'   FID-50K = {van_fid:.3f}'
        if rtr_fid is not None:
            rtr_label += f'   FID-50K = {rtr_fid:.3f}'
        draw.text((10, 10), van_label, font=FONT, fill='black')
        draw.text((w + gap + 10, 10), rtr_label, font=FONT, fill='black')

        out_path = COMPARISONS_DIR / f'compare_step{step:02d}_seed{seed}.png'
        canvas.save(out_path)
        made += 1

print(f'Composed {made} comparison figures, skipped {skipped} (missing grids).')
print(f'Output: {COMPARISONS_DIR}')

preview = COMPARISONS_DIR / 'compare_step16_seed0.png'
if preview.exists():
    from IPython.display import Image as IPImage, display
    display(IPImage(str(preview)))
